In [0]:
from datetime import datetime
from zoneinfo import ZoneInfo
import os
import pandas as pd

In [ ]:
# 定数
S3_BUCKET_PATH = {
    "dev" : "s3://data-exchange-common-from-databricks-test/",
    "prod": "s3://data-exchange-common-from-databricks/"
}

DATABRICKS_CATALOG = {
    "dev" : "dataplatform_public_published_dev",
    "prod": "dataplatform_public_published"
}
DATABRICKS_SCHEMA = "data_exchange_common"

In [ ]:
# 日時を取得
now_jst = datetime.now(ZoneInfo("Asia/Tokyo")).strftime("%Y%m%d_%H%M%S")
print(now_jst) 

In [ ]:
# 引数の取得
dbutils.widgets.text("OUTPUT_TABLE", "")
output_table = dbutils.widgets.get("OUTPUT_TABLE")

stage = os.environ.get("STAGE")

# テーブル名
table_path = f"{DATABRICKS_CATALOG[stage]}.{DATABRICKS_SCHEMA }.{output_table}"

# S3の出力先
output_dir = f"{S3_BUCKET_PATH[stage]}{now_jst}_{output_table}"

print(f"出力対象テーブル：{table_path}\nS3の出力先：{output_dir}")

In [ ]:
try:
    df = spark.table(table_path)
except Exception as e:
    raise Exception(f"テーブルの取得に失敗しました。テーブル名：{table_path}\nError: {e}")

In [ ]:
try:
    df.coalesce(1).write.mode("overwrite").option("header", "true").option("compression", "gzip").csv(output_dir)
except Exception as e:
    raise Exception(f"テーブルの出力に失敗しました。テーブル名：{table_path}\n出力先：{output_dir}\nError: {e}")